# Phase 4 — Backtest

Five strategies on Test period 2024-07-01 → 2026-04-21. All straddle strategies share the same trade identity (Phase 2 `(strike, expiration)`) — Phase 4 **re-prices** them with realistic frictions, it does not re-select.

Stress grid:
- slippage `k ∈ {0.5 retail-realistic, 1 conservative-realistic, 2 stress}`
- notional `∈ {$1K, $10K base, $50K}`
- drop-best-month / drop-worst-month per strategy

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

HERE = Path('.').resolve()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import backtest as bt

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Run the full backtest (base + stress + robustness)

In [ ]:
result = bt.main()
stress_df = result['stress_df']
out_base  = result['out_base']
recon     = result['recon']
vrp_thr   = result['vrp_threshold']

## 2. Reconciliation diagnostic

Phase 4 re-prices Phase 2's trades. Gross P&L at **mid/mid** pricing in Phase 4 must reconcile to `pnl_pct_5d × premium_collected × 100` from Phase 2. Any deviation beyond float rounding indicates a bug.

In [ ]:
print(f'Reconciliation (1-contract $ basis, mid-priced):')
print(f'  n rows           : {recon["n"]}')
print(f'  max abs $ diff   : ${recon["max_abs_diff"]:.6f}')
print(f'  mean abs $ diff  : ${recon["mean_abs_diff"]:.6f}')
print(f'  max rel diff     : {recon["max_rel_diff"]:.2e}')
if recon['max_abs_diff'] < 0.01:
    print('  -> PASS (within float rounding)')
else:
    print('  -> FLAG: inspect load-join or fill logic')

## 3. Base-config summary (k=1, $10K, all months)

In [ ]:
base = stress_df[stress_df['config'] == 'base_k1_10K_allmonths'].set_index('strategy')
cols_summary = ['terminal_equity','total_return_pct','annualized_return',
                'sharpe_non_overlap','sortino','max_dd_dollar','max_dd_pct','calmar',
                'win_rate','profit_factor','mean_trade_dollar','best_trade_dollar',
                'worst_trade_dollar','n_trades','capital_efficiency',
                'peak_margin_util_pct','n_margin_rejects','n_missing_quote',
                'n_wide_spread_trades']
cols_summary = [c for c in cols_summary if c in base.columns]
with pd.option_context('display.max_columns', None, 'display.width', 260):
    print(base[cols_summary].round(3).to_string())

## 4. Stress tests — slippage multiplier

In [ ]:
pivot_cols = ['terminal_equity','total_return_pct','sharpe_non_overlap','sortino','max_dd_pct','win_rate','capital_efficiency']
cfgs_slip = ['k0.5_10K_allmonths', 'base_k1_10K_allmonths', 'k2_10K_allmonths']
for c in cfgs_slip:
    label = {'k0.5_10K_allmonths':'k=0.5 (retail-realistic)',
             'base_k1_10K_allmonths':'k=1 (conservative-realistic, BASE)',
             'k2_10K_allmonths':'k=2 (stress: filling through book)'}[c]
    print(f'\n--- {label} ---')
    sub = stress_df[stress_df['config']==c].set_index('strategy')[pivot_cols]
    print(sub.round(3).to_string())

# Ranking flip check on sharpe_non_overlap
print('\n--- Ranking on Sharpe (non-overlap): ---')
for c in cfgs_slip:
    sub = stress_df[stress_df['config']==c].set_index('strategy')['sharpe_non_overlap']
    order = sub.dropna().sort_values(ascending=False).index.tolist()
    print(f'  {c}: {order}')

## 5. Stress tests — position size

In [ ]:
cfgs_sz = ['k1_1K_allmonths', 'base_k1_10K_allmonths', 'k1_50K_allmonths']
for c in cfgs_sz:
    label = {'k1_1K_allmonths':'$1K notional',
             'base_k1_10K_allmonths':'$10K notional (BASE)',
             'k1_50K_allmonths':'$50K notional'}[c]
    print(f'\n--- {label} ---')
    sub = stress_df[stress_df['config']==c].set_index('strategy')[
        pivot_cols + ['peak_margin_util_pct','n_margin_rejects']]
    print(sub.round(3).to_string())

print('\n--- Ranking on Sharpe (non-overlap): ---')
for c in cfgs_sz:
    sub = stress_df[stress_df['config']==c].set_index('strategy')['sharpe_non_overlap']
    order = sub.dropna().sort_values(ascending=False).index.tolist()
    print(f'  {c}: {order}')

## 6. Single-month robustness

For each strategy, drop the single best and single worst calendar month in the base run and rerun. If a strategy's edge depends on a single outlier month, terminal equity / Sharpe will collapse.

In [ ]:
for label in ('drop_best', 'drop_worst'):
    key = f'k1_10K_{label}'
    print(f'\n--- k=1, $10K, {label} month dropped per strategy ---')
    sub = stress_df[stress_df['config']==key].set_index('strategy')[
        ['dropped_month'] + pivot_cols]
    print(sub.round(3).to_string())

## 7. Equity curves (log-scale overlay)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
for strat, color in [('always_short', 'tab:red'),
                     ('vrp_rule',     'tab:purple'),
                     ('logreg',       'tab:blue'),
                     ('lgbm',         'tab:orange'),
                     ('buy_hold_spy', 'tab:green')]:
    eq = out_base[strat]['daily']['equity']
    term = eq.iloc[-1]
    ax.plot(eq.index, eq.values, label=f'{strat} (end=${term:,.0f})', color=color, lw=1.3)
ax.axhline(100_000, color='black', ls='--', lw=0.6, alpha=0.5)
ax.set_yscale('log')
ax.set_ylabel('Equity ($, log scale)')
ax.set_title('Equity curves — base config (k=1, $10K, all months)')
for d_label, d in [('Yen carry', '2024-08-05'), ('Liberation Day', '2025-04-08')]:
    ax.axvline(pd.Timestamp(d), color='red', alpha=0.3, ls='-.')
    ax.text(pd.Timestamp(d), ax.get_ylim()[0]*1.1, d_label, rotation=90, va='bottom', ha='right', fontsize=8, color='red')
ax.legend(loc='lower left')
plt.tight_layout(); plt.show()

## 8. Daily capital deployed (margin $) over time

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for strat, color in [('always_short', 'tab:red'),
                     ('vrp_rule',     'tab:purple'),
                     ('logreg',       'tab:blue'),
                     ('lgbm',         'tab:orange')]:
    d = out_base[strat]['daily']['deployed_margin']
    ax.plot(d.index, d.values, label=strat, color=color, lw=1.1)
ax.set_ylabel('Deployed margin ($)')
ax.set_title('Daily capital deployed per strategy — base config')
ax.legend()
plt.tight_layout(); plt.show()

## 9. Underwater drawdown curves

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for strat, color in [('always_short', 'tab:red'),
                     ('vrp_rule',     'tab:purple'),
                     ('logreg',       'tab:blue'),
                     ('lgbm',         'tab:orange'),
                     ('buy_hold_spy', 'tab:green')]:
    eq = out_base[strat]['daily']['equity']
    peak = eq.cummax()
    dd   = (eq - peak) / peak * 100
    ax.plot(dd.index, dd.values, label=strat, color=color, lw=1.1)
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('Drawdown (%)')
ax.set_title('Underwater curves — base config')
ax.legend()
plt.tight_layout(); plt.show()

## 10. Per-trade P&L histogram

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, strat in zip(axes.ravel(), ['always_short','vrp_rule','logreg','lgbm']):
    t = out_base[strat]['trades']
    if t.empty: continue
    net = t['net_pnl'].to_numpy()
    ax.hist(net, bins=40, color='tab:blue', alpha=0.7)
    ax.axvline(0, color='black', lw=0.8)
    ax.axvline(float(np.median(net)), color='red', lw=0.8, ls='--', label=f'median ${np.median(net):,.0f}')
    ax.set_title(f'{strat}  (n={len(t)}, mean=${net.mean():,.0f}, worst=${net.min():,.0f})')
    ax.set_xlabel('Net P&L per trade ($)')
    ax.legend()
fig.suptitle('Per-trade net P&L distribution — base config')
plt.tight_layout(); plt.show()

## 11. Monthly returns heatmap — winning strategy (VRP rule)

In [ ]:
winner = stress_df[stress_df['config']=='base_k1_10K_allmonths'].sort_values('sharpe_non_overlap', ascending=False).iloc[0]['strategy']
print(f'Winner by non-overlap Sharpe: {winner}')
t = out_base[winner]['trades'].copy()
t['exit_dt'] = pd.to_datetime(t['exit_date'])
t['y'] = t['exit_dt'].dt.year
t['m'] = t['exit_dt'].dt.month
monthly = t.groupby(['y','m'])['net_pnl'].sum().unstack('m')
monthly = monthly.reindex(columns=range(1,13))

fig, ax = plt.subplots(figsize=(11, 3 + 0.6 * len(monthly)))
im = ax.imshow(monthly.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(12)); ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_yticks(range(len(monthly.index))); ax.set_yticklabels(monthly.index)
for (i, j), v in np.ndenumerate(monthly.values):
    if np.isfinite(v):
        ax.text(j, i, f'{v:,.0f}', ha='center', va='center',
                color='black', fontsize=8)
plt.colorbar(im, ax=ax, label='Monthly $ P&L')
ax.set_title(f'Monthly P&L heatmap — {winner} (base config)')
plt.tight_layout(); plt.show()

## 12. Capital efficiency comparison

`cumulative net P&L / Σ daily deployed-capital $` — the selectivity-adjusted metric.  A low-coverage strategy that idles capital but earns when deployed should win on this axis.

In [ ]:
bar = stress_df[stress_df['config']=='base_k1_10K_allmonths'].set_index('strategy')['capital_efficiency']
bar = bar.dropna().sort_values()
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['tab:red' if x <= 0 else 'tab:green' for x in bar.values]
ax.barh(bar.index, bar.values * 10_000, color=colors)   # scale to readable units: pnl per $10K-day
ax.set_xlabel('Net P&L per $10K-day deployed ($)')
ax.set_title('Capital efficiency — base config')
plt.tight_layout(); plt.show()

## 13. 10d-target robustness cross-check

Phase 3 showed VRP rule wins on `profitable_5d` (primary target). Here we verify the same ranking on `profitable_10d` — same Phase 4 trade mechanics but the VRP threshold and trained models are re-derived on the 10d target.

In [ ]:
# Pull the 10d summary from the ml_pipeline --secondary run's already-saved metrics
try:
    import subprocess, os, json
    # The --secondary run already executed and saved; report its metrics.csv aggregates
    # directly here as a cross-check.
    print('Phase 3 (profitable_10d) trade-econ summary — Test split, from metrics.csv after --secondary run:')
    import ml_pipeline as mp_mod
    r10 = mp_mod.run_pipeline(target='profitable_10d', pnl_col='pnl_pct_10d', horizon_days=10, save=False)
    m10 = r10['metrics_df']
    print(m10[m10['split']=='test'].set_index('model')[
        ['accuracy','mcc','coverage_pct','hit_rate_taken','mean_pnl_pct_taken',
         'sharpe_non_overlapping','pct_disasters_avoided','mean_pnl_slipped']].round(4).to_string())
    # ranking on non-overlapping sharpe
    rank = m10[m10['split']=='test'].set_index('model')['sharpe_non_overlapping'].dropna().sort_values(ascending=False)
    print('\n10d-target Sharpe (non-overlap) ranking on Test:')
    print(rank.to_string())
except Exception as e:
    print(f'10d cross-check skipped: {e}')